#### Silver Layer – Data Cleaning & Conformance

- Parse timestamps → timestamp
- Filter bad rows (e.g., invalid customer_id)
- Deduplicate on business keys
- Build a slowly changing dimension-style customer table using MERG

#### Create Silver tables 

In [0]:
use catalog project;

CREATE TABLE IF NOT EXISTS silver.orders_clean (
  order_id      BIGINT,
  order_ts      TIMESTAMP,
  order_date    DATE,
  customer_id   BIGINT,
  product_id    STRING,
  category      STRING,
  qty           INT,
  price         DECIMAL(10,2),
  status        STRING,
  load_ts       TIMESTAMP
) USING DELTA;

-- Customer dimension (Type 1 SCD style)
CREATE TABLE IF NOT EXISTS silver.customers_dim (
  customer_id    BIGINT,
  customer_name  STRING,
  email          STRING,
  city           STRING,
  state          STRING,
  country        STRING,
  segment        STRING,
  last_update_ts TIMESTAMP,
  load_ts        TIMESTAMP
) USING DELTA;


In [0]:
select * from silver.customers_dim;

customer_id,customer_name,email,city,state,country,segment,last_update_ts,load_ts


#### Transform Bronze to Silver: Orders 

In [0]:
%python

from pyspark.sql.functions import col, to_timestamp, to_date, row_number, current_timestamp
from pyspark.sql.window import Window

bronze_orders = spark.table("bronze.orders_raw")

# 1) Convert timestamp & derive order_date
orders_transformed = (
    bronze_orders
    .withColumn("order_ts_parsed", to_timestamp("order_ts"))
    .withColumn("order_date", to_date("order_ts_parsed"))
)

# 2) Filter invalid rows
#   - order_ts null
#   - qty <= 0
#   - price <= 0
orders_filtered = (
    orders_transformed
    .filter(col("order_ts_parsed").isNotNull())
    .filter(col("qty") > 0)
    .filter(col("price") > 0)
)

# 3) Deduplicate by (order_id, customer_id, order_ts_parsed)
window_spec = Window.partitionBy("order_id", "customer_id", "order_ts_parsed") \
                    .orderBy(col("ingestion_ts").desc())

orders_deduped = (
    orders_filtered
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

# 4) Select & rename final columns
orders_clean_df = (
    orders_deduped
    .select(
        col("order_id"),
        col("order_ts_parsed").alias("order_ts"),
        col("order_date"),
        col("customer_id"),
        col("product_id"),
        col("category"),
        col("qty"),
        col("price"),
        col("status"),
    )
    .withColumn("load_ts", current_timestamp())
)

# 5) Write to silver.orders_clean (append)
orders_clean_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.orders_clean")


In [0]:
select * from project.silver.orders_clean;

order_id,order_ts,order_date,customer_id,product_id,category,qty,price,status,load_ts
1001,2025-01-01T10:15:00Z,2025-01-01,1,P01,Electronics,1,500.00,PAID,2025-12-13T07:03:37.982289Z
1002,2025-01-01T10:18:00Z,2025-01-01,2,P02,Books,2,250.00,PAID,2025-12-13T07:03:37.982289Z
1003,2025-01-01T10:20:00Z,2025-01-01,1,P03,Electronics,1,1200.00,CANCELLED,2025-12-13T07:03:37.982289Z
1004,2025-01-01T10:22:00Z,2025-01-01,3,P02,Books,1,125.00,PAID,2025-12-13T07:03:37.982289Z
1005,2025-01-01T10:25:00Z,2025-01-01,4,P04,Toys,2,300.00,PAID,2025-12-13T07:03:37.982289Z
1006,2025-01-02T09:00:00Z,2025-01-02,2,P01,Electronics,1,500.00,PAID,2025-12-13T07:03:37.982289Z
1007,2025-01-02T09:05:00Z,2025-01-02,5,P05,Fashion,3,700.00,PAID,2025-12-13T07:03:37.982289Z
1008,2025-01-02T09:10:00Z,2025-01-02,999,P06,Electronics,1,800.00,PAID,2025-12-13T07:03:37.982289Z


In [0]:
describe table project.silver.orders_clean

col_name,data_type,comment
order_id,bigint,null
order_ts,timestamp,null
order_date,date,null
customer_id,bigint,null
product_id,string,null
category,string,null
qty,int,null
price,"decimal(10,2)",null
status,string,null
load_ts,timestamp,null


#### Transform Bronze to Silver: Customers (Type-1 SCD with MERGE)

In [0]:
%python
from pyspark.sql.functions import to_timestamp

bronze_customers = spark.table("bronze.customers_raw")

customers_stage = (
    bronze_customers
    .withColumn("update_ts_parsed", to_timestamp("update_ts"))
    .select(
        "customer_id",
        "customer_name",
        "email",
        "city",
        "state",
        "country",
        "segment",
        col("update_ts_parsed").alias("last_update_ts")
    )
    .withColumn("load_ts", current_timestamp())
)

customers_stage.createOrReplaceTempView("customers_stage")


In [0]:
select * from customers_stage;

customer_id,customer_name,email,city,state,country,segment,last_update_ts,load_ts
1,Amit Kumar,amit@example.com,Chennai,TN,India,RET,2025-01-01T00:00:00Z,2025-12-13T07:07:55.375017Z
2,Latha Rao,latha@example.com,Bangalore,KA,India,RET,2025-01-01T00:00:00Z,2025-12-13T07:07:55.375017Z
3,Rahul Jain,rahul@example.com,Hyderabad,TS,India,RET,2025-01-01T00:00:00Z,2025-12-13T07:07:55.375017Z
4,Meena Iyer,meena@example.com,Mumbai,MH,India,ENT,2025-01-01T00:00:00Z,2025-12-13T07:07:55.375017Z
2,Latha R,latha.r@example.com,Bangalore,KA,India,ENT,2025-01-02T00:00:00Z,2025-12-13T07:07:55.375017Z
5,Arjun Dev,arjun@example.com,Delhi,DL,India,RET,2025-01-02T00:00:00Z,2025-12-13T07:07:55.375017Z


#### Use SQL MERGE to apply updates & inserts to silver.customers_dim:

In [0]:
MERGE INTO project.silver.customers_dim AS tgt
USING customers_stage AS src
ON tgt.customer_id = src.customer_id

WHEN MATCHED AND src.last_update_ts > tgt.last_update_ts THEN
  UPDATE SET
    tgt.customer_name  = src.customer_name,
    tgt.email          = src.email,
    tgt.city           = src.city,
    tgt.state          = src.state,
    tgt.country        = src.country,
    tgt.segment        = src.segment,
    tgt.last_update_ts = src.last_update_ts,
    tgt.load_ts        = src.load_ts

WHEN NOT MATCHED THEN
  INSERT (
    customer_id,
    customer_name,
    email,
    city,
    state,
    country,
    segment,
    last_update_ts,
    load_ts
  )
  VALUES (
    src.customer_id,
    src.customer_name,
    src.email,
    src.city,
    src.state,
    src.country,
    src.segment,
    src.last_update_ts,
    src.load_ts
  );


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
select * from project.silver.customers_dim

customer_id,customer_name,email,city,state,country,segment,last_update_ts,load_ts
1,Amit Kumar,amit@example.com,Chennai,TN,India,RET,2025-01-01T00:00:00Z,2025-12-13T07:11:09.326257Z
2,Latha Rao,latha@example.com,Bangalore,KA,India,RET,2025-01-01T00:00:00Z,2025-12-13T07:11:09.326257Z
3,Rahul Jain,rahul@example.com,Hyderabad,TS,India,RET,2025-01-01T00:00:00Z,2025-12-13T07:11:09.326257Z
4,Meena Iyer,meena@example.com,Mumbai,MH,India,ENT,2025-01-01T00:00:00Z,2025-12-13T07:11:09.326257Z
2,Latha R,latha.r@example.com,Bangalore,KA,India,ENT,2025-01-02T00:00:00Z,2025-12-13T07:11:09.326257Z
5,Arjun Dev,arjun@example.com,Delhi,DL,India,RET,2025-01-02T00:00:00Z,2025-12-13T07:11:09.326257Z
